## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os

# Config

In [ ]:
from config import (
    FEATURE_COLS,
    SKLEARN_INFERENCE_MODEL_PICKLE_PATH,
    SKLEARN_INFERENCE_INPUT_PATH,
    SKLEARN_INFERENCE_SCALER_JOBLIB_PATH,
    SKLEARN_INFERENCE_INPUT_LABEL_ENCODING_PATH,
    SKLEARN_INFERENCE_EXPECTED_PATH,
    SKLEARN_INFERENCE_DIR,
    SKLEARN_INFERENCE_OUTPUT_PATH,
)

## Load the Pickle Model

In [ ]:
model_path = SKLEARN_INFERENCE_MODEL_PICKLE_PATH

with open(model_path, 'rb') as f:
    model = pickle.load(f)
print(f'Model loaded from: {model_path}')

## Load Inference Input Data

In [ ]:
input_path = SKLEARN_INFERENCE_INPUT_PATH

df_input = pd.read_csv(input_path)
print(f'Input data shape: {df_input.shape}')
df_input.head()

In [ ]:
# Convert columns to float
df_input[FEATURE_COLS] = df_input[FEATURE_COLS].astype(float)

In [ ]:
# Display the encoded columns
print("Columns after:")
print(df_input.columns.tolist())

In [ ]:
df_input

# Load Scaler

In [ ]:
scaler_path = SKLEARN_INFERENCE_SCALER_JOBLIB_PATH

import joblib
scaler = joblib.load(scaler_path)

# Load Label Encoding

In [ ]:
encoding_path = SKLEARN_INFERENCE_INPUT_LABEL_ENCODING_PATH

import joblib
# To convert back during inference:
le = joblib.load(encoding_path)

## Prepare Features for Inference

In [ ]:
# Adjust feature columns as needed to match training
feature_cols = FEATURE_COLS
X_infer = df_input[feature_cols].values.astype(np.float32)
print(f'Inference features shape: {X_infer.shape}')

# standardize the features

In [ ]:
x_infer_scaled = scaler.transform(X_infer)

## Run Inference

In [ ]:
y_pred = model.predict(x_infer_scaled)
print('Predictions:', y_pred)

original_species = le.inverse_transform(y_pred)
print('Original Species:', original_species)

# Verify

In [ ]:
expected_path = SKLEARN_INFERENCE_EXPECTED_PATH

df_expected = pd.read_csv(expected_path)
expected_species = df_expected['species'].to_numpy(dtype=str)
orig = np.asarray(original_species, dtype=str)

if np.array_equal(orig, expected_species):
    print('Verification: correct — predictions match expected.csv')
else:
    print('Verification: incorrect — predictions do not match expected.csv')
    print('Expected:', expected_species)
    print('Got:', orig)

# Prepare Directory

In [ ]:
output_dir = SKLEARN_INFERENCE_DIR

os.makedirs(output_dir, exist_ok=True)

## Save Predictions to CSV

In [ ]:
output_csv = SKLEARN_INFERENCE_OUTPUT_PATH

df_output = df_input.copy()
df_output['Prediction'] = y_pred
df_output['Species'] = original_species
df_output.to_csv(output_csv, index=False)
print(f'Predictions saved to: {output_csv}')